# 10 Signal Performance Analysis v0.1

Diagnostics notebook for priced trade-candidate results from notebook `09_trade_candidate_pricing_v0_1`.

Scope:
- Analyze entry-to-expiry, same-notional normalized P&L.
- Compare performance by side, tenor, tenor group, signal tier, signal buckets, RSI bucket, realized-vol bucket, and year.
- Diagnose front-tenor put underperformance.
- Save audit outputs to `data/audit`.

This notebook does **not** introduce new trading rules, optimization, risk sizing, or portfolio simulation.


In [ ]:
# ============================================================
# 10_signal_performance_analysis_v0_1
#
# Goal:
#   Analyze priced trade-candidate results from notebook 09.
#
# Scope:
#   Diagnostics only.
#   No new trading rules.
#   No optimization.
#   No risk sizing.
#   No portfolio simulation.
#
# Main question:
#   Which signal dimensions explain performance differences,
#   especially weak front-tenor put performance?
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PRICING_INPUT_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_pricing_v0_1.parquet"

SIGNAL_PERFORMANCE_SUMMARY_CSV = AUDIT_DIR / "signal_performance_summary_v0_1.csv"
SIGNAL_PERFORMANCE_TENOR_TIER_CSV = AUDIT_DIR / "signal_performance_tenor_tier_v0_1.csv"
SIGNAL_PERFORMANCE_BUCKETS_CSV = AUDIT_DIR / "signal_performance_buckets_v0_1.csv"
SIGNAL_PERFORMANCE_TAILS_CSV = AUDIT_DIR / "signal_performance_tail_losses_v0_1.csv"

print("Project root:", PROJECT_ROOT)
print("Pricing input exists:", PRICING_INPUT_PARQUET.exists())

In [ ]:
# ============================================================
# Load priced trade-candidate results from notebook 09
# ============================================================

pricing_df = pd.read_parquet(PRICING_INPUT_PARQUET).copy()

required_cols = [
    "trade_date",
    "trade_side",
    "selected_tenor",
    "actual_dte",
    "signal_tier",
    "signal_state",
    "pricing_status",
    "entry_credit_points",
    "short_option_pnl_points",
    "short_option_win",
    "short_option_expired_otm",
    "normalized_pnl_dollars",
    "normalized_pnl_pct_notional",
    "signal_primary_vrp",
    "signal_blended_z",
    "signal_3m_z",
    "signal_1y_z",
    "signal_implied_vol",
    "signal_trailing_rv",
    "spx_rsi_14",
    "spx_close",
    "expiry_spx_close",
    "short_strike_selected",
]

missing_cols = [c for c in required_cols if c not in pricing_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

pricing_df["trade_date"] = pricing_df["trade_date"].astype(int)
pricing_df["trade_year"] = pricing_df["trade_date"].astype(str).str[:4].astype(int)

priced_df = pricing_df[pricing_df["pricing_status"] == "priced"].copy()
not_priced_df = pricing_df[pricing_df["pricing_status"] != "priced"].copy()

put_df = priced_df[priced_df["trade_side"] == "put"].copy()
call_df = priced_df[priced_df["trade_side"] == "call"].copy()

print("Total rows:", len(pricing_df))
print("Priced rows:", len(priced_df))
print("Not priced rows:", len(not_priced_df))

print("\nPriced rows by side:")
display(priced_df["trade_side"].value_counts().rename("count").to_frame())

print("\nDate range:")
print(priced_df["trade_date"].min(), "to", priced_df["trade_date"].max())

display(priced_df.tail(20))

In [ ]:
# ============================================================
# Create analysis buckets
# ============================================================

def make_tenor_group(tenor):
    if pd.isna(tenor):
        return np.nan

    tenor = float(tenor)

    if tenor <= 18:
        return "front_9_18d"

    if tenor <= 27:
        return "middle_21_27d"

    return "back_30_33d"


priced_df["tenor_group"] = priced_df["selected_tenor"].apply(make_tenor_group)

# Buckets chosen to line up with existing put thresholds and useful diagnostics.
priced_df["blended_z_bucket"] = pd.cut(
    priced_df["signal_blended_z"],
    bins=[-np.inf, 0.1, 0.65, 0.7, 1.0, 1.5, np.inf],
    labels=[
        "<=0.10",
        "0.10_to_0.65",
        "0.65_to_0.70",
        "0.70_to_1.00",
        "1.00_to_1.50",
        ">1.50",
    ],
)

priced_df["primary_vrp_bucket"] = pd.cut(
    priced_df["signal_primary_vrp"],
    bins=[-np.inf, 0.7, 0.85, 1.0, 1.25, 1.5, np.inf],
    labels=[
        "<=0.70",
        "0.70_to_0.85",
        "0.85_to_1.00",
        "1.00_to_1.25",
        "1.25_to_1.50",
        ">1.50",
    ],
)

priced_df["trailing_rv_bucket"] = pd.cut(
    priced_df["signal_trailing_rv"],
    bins=[-np.inf, 6.8, 8.5, 12.0, 16.0, 25.0, np.inf],
    labels=[
        "<=6.8",
        "6.8_to_8.5",
        "8.5_to_12",
        "12_to_16",
        "16_to_25",
        ">25",
    ],
)

priced_df["rsi_bucket"] = pd.cut(
    priced_df["spx_rsi_14"],
    bins=[-np.inf, 40, 50, 58, 65, 70, 72, 77, np.inf],
    labels=[
        "<=40",
        "40_to_50",
        "50_to_58",
        "58_to_65",
        "65_to_70",
        "70_to_72",
        "72_to_77",
        ">77",
    ],
)

# Refresh side-specific frames after adding buckets.
put_df = priced_df[priced_df["trade_side"] == "put"].copy()
call_df = priced_df[priced_df["trade_side"] == "call"].copy()

print("Tenor group counts:")
display(
    priced_df
    .groupby(["trade_side", "tenor_group"])
    .size()
    .unstack(fill_value=0)
)

print("\nPut tier counts:")
display(put_df["signal_tier"].value_counts().rename("count").to_frame())

In [ ]:
# ============================================================
# Reusable performance summary helper
# ============================================================

def summarize_performance(df, group_cols):
    """
    Summarize entry-to-expiry short-option performance.

    Normalized P&L is same-notional P&L from notebook 09.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    summary = (
        df
        .groupby(group_cols, dropna=False, observed=False)
        .agg(
            rows=("trade_date", "count"),
            win_rate=("short_option_win", "mean"),
            expired_otm_rate=("short_option_expired_otm", "mean"),
            avg_entry_credit_points=("entry_credit_points", "mean"),
            median_entry_credit_points=("entry_credit_points", "median"),
            avg_pnl_points=("short_option_pnl_points", "mean"),
            median_pnl_points=("short_option_pnl_points", "median"),
            worst_pnl_points=("short_option_pnl_points", "min"),
            best_pnl_points=("short_option_pnl_points", "max"),
            avg_normalized_pnl_pct=("normalized_pnl_pct_notional", "mean"),
            median_normalized_pnl_pct=("normalized_pnl_pct_notional", "median"),
            worst_normalized_pnl_pct=("normalized_pnl_pct_notional", "min"),
            best_normalized_pnl_pct=("normalized_pnl_pct_notional", "max"),
            avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
            median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
            worst_normalized_pnl_dollars=("normalized_pnl_dollars", "min"),
            best_normalized_pnl_dollars=("normalized_pnl_dollars", "max"),
            avg_signal_primary_vrp=("signal_primary_vrp", "mean"),
            avg_signal_blended_z=("signal_blended_z", "mean"),
            avg_signal_trailing_rv=("signal_trailing_rv", "mean"),
            avg_signal_implied_vol=("signal_implied_vol", "mean"),
            avg_rsi=("spx_rsi_14", "mean"),
            avg_actual_dte=("actual_dte", "mean"),
        )
        .reset_index()
    )

    return summary

In [ ]:
# ============================================================
# High-level performance summaries
# ============================================================

summary_by_side_df = summarize_performance(
    priced_df,
    ["trade_side"],
)

summary_by_side_year_df = summarize_performance(
    priced_df,
    ["trade_side", "trade_year"],
)

summary_by_side_tenor_df = summarize_performance(
    priced_df,
    ["trade_side", "selected_tenor"],
)

summary_by_side_tenor_group_df = summarize_performance(
    priced_df,
    ["trade_side", "tenor_group"],
)

print("Summary by side:")
display(summary_by_side_df)

print("\nSummary by side and tenor:")
display(summary_by_side_tenor_df)

print("\nSummary by side and tenor group:")
display(summary_by_side_tenor_group_df)

print("\nSummary by side and year:")
display(summary_by_side_year_df)

In [ ]:
# ============================================================
# Put tier and tenor x tier diagnostics
# ============================================================

put_tier_summary_df = summarize_performance(
    put_df,
    ["signal_tier"],
).sort_values("signal_tier")

put_tenor_tier_summary_df = summarize_performance(
    put_df,
    ["selected_tenor", "signal_tier"],
).sort_values(["selected_tenor", "signal_tier"])

put_tenor_group_tier_summary_df = summarize_performance(
    put_df,
    ["tenor_group", "signal_tier"],
).sort_values(["tenor_group", "signal_tier"])

print("Put tier summary:")
display(put_tier_summary_df)

print("\nPut tenor x tier summary:")
display(put_tenor_tier_summary_df)

print("\nPut tenor group x tier summary:")
display(put_tenor_group_tier_summary_df)

In [ ]:
# ============================================================
# Signal bucket diagnostics
# ============================================================

put_blended_z_bucket_summary_df = summarize_performance(
    put_df,
    ["blended_z_bucket"],
)

put_primary_vrp_bucket_summary_df = summarize_performance(
    put_df,
    ["primary_vrp_bucket"],
)

put_trailing_rv_bucket_summary_df = summarize_performance(
    put_df,
    ["trailing_rv_bucket"],
)

put_rsi_bucket_summary_df = summarize_performance(
    put_df,
    ["rsi_bucket"],
)

print("Put performance by blended z bucket:")
display(put_blended_z_bucket_summary_df)

print("\nPut performance by primary VRP bucket:")
display(put_primary_vrp_bucket_summary_df)

print("\nPut performance by trailing RV bucket:")
display(put_trailing_rv_bucket_summary_df)

print("\nPut performance by RSI bucket:")
display(put_rsi_bucket_summary_df)

In [ ]:
# ============================================================
# Front-tenor failure diagnostics
# ============================================================

front_put_df = put_df[put_df["tenor_group"] == "front_9_18d"].copy()
nonfront_put_df = put_df[put_df["tenor_group"] != "front_9_18d"].copy()

front_vs_nonfront_summary_df = summarize_performance(
    put_df,
    ["tenor_group"],
)

front_put_tier_summary_df = summarize_performance(
    front_put_df,
    ["signal_tier"],
).sort_values("signal_tier")

front_put_z_bucket_summary_df = summarize_performance(
    front_put_df,
    ["blended_z_bucket"],
)

front_put_rv_bucket_summary_df = summarize_performance(
    front_put_df,
    ["trailing_rv_bucket"],
)

front_put_rsi_bucket_summary_df = summarize_performance(
    front_put_df,
    ["rsi_bucket"],
)

print("Front vs non-front put summary:")
display(front_vs_nonfront_summary_df)

print("\nFront put performance by tier:")
display(front_put_tier_summary_df)

print("\nFront put performance by blended z bucket:")
display(front_put_z_bucket_summary_df)

print("\nFront put performance by trailing RV bucket:")
display(front_put_rv_bucket_summary_df)

print("\nFront put performance by RSI bucket:")
display(front_put_rsi_bucket_summary_df)

In [ ]:
# ============================================================
# Tail-loss diagnostics
# ============================================================

put_df["loss_bucket"] = pd.cut(
    put_df["normalized_pnl_pct_notional"],
    bins=[-np.inf, -0.10, -0.05, -0.02, 0.0, 0.01, 0.02, np.inf],
    labels=[
        "<=-10%",
        "-10%_to_-5%",
        "-5%_to_-2%",
        "-2%_to_0%",
        "0%_to_1%",
        "1%_to_2%",
        ">2%",
    ],
)

put_loss_bucket_summary_df = summarize_performance(
    put_df,
    ["loss_bucket"],
)

put_tail_loss_df = (
    put_df
    .sort_values("normalized_pnl_pct_notional")
    [
        [
            "trade_date",
            "trade_year",
            "selected_expiry_date",
            "selected_tenor",
            "tenor_group",
            "actual_dte",
            "signal_tier",
            "signal_state",
            "spx_close",
            "expiry_spx_close",
            "short_strike_selected",
            "entry_credit_points",
            "expiry_intrinsic_value",
            "short_option_pnl_points",
            "normalized_pnl_dollars",
            "normalized_pnl_pct_notional",
            "signal_primary_vrp",
            "signal_blended_z",
            "signal_3m_z",
            "signal_1y_z",
            "signal_trailing_rv",
            "signal_implied_vol",
            "spx_rsi_14",
        ]
    ]
    .head(50)
)

print("Put loss bucket summary:")
display(put_loss_bucket_summary_df)

print("\nWorst 50 put losses:")
display(put_tail_loss_df)

In [ ]:
# ============================================================
# Expectancy decomposition
# ============================================================

def expectancy_decomposition(df, group_cols):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    working = df.copy()

    working["is_win"] = working["short_option_pnl_points"] > 0
    working["is_loss"] = working["short_option_pnl_points"] <= 0

    grouped = working.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for group_key, group in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        wins = group[group["is_win"]].copy()
        losses = group[group["is_loss"]].copy()

        win_rate = len(wins) / len(group) if len(group) > 0 else np.nan
        loss_rate = len(losses) / len(group) if len(group) > 0 else np.nan

        avg_win_pct = wins["normalized_pnl_pct_notional"].mean()
        avg_loss_pct = losses["normalized_pnl_pct_notional"].mean()

        expectancy_pct = group["normalized_pnl_pct_notional"].mean()

        row = {
            col: value
            for col, value in zip(group_cols, group_key)
        }

        row.update({
            "rows": len(group),
            "win_rate": win_rate,
            "loss_rate": loss_rate,
            "avg_win_pct": avg_win_pct,
            "avg_loss_pct": avg_loss_pct,
            "expectancy_pct": expectancy_pct,
            "win_contribution_pct": win_rate * avg_win_pct if pd.notna(avg_win_pct) else np.nan,
            "loss_contribution_pct": loss_rate * avg_loss_pct if pd.notna(avg_loss_pct) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


put_expectancy_by_tenor_df = expectancy_decomposition(
    put_df,
    ["selected_tenor"],
)

put_expectancy_by_tier_df = expectancy_decomposition(
    put_df,
    ["signal_tier"],
)

put_expectancy_by_tenor_group_tier_df = expectancy_decomposition(
    put_df,
    ["tenor_group", "signal_tier"],
)

print("Put expectancy by tenor:")
display(put_expectancy_by_tenor_df)

print("\nPut expectancy by tier:")
display(put_expectancy_by_tier_df)

print("\nPut expectancy by tenor group and tier:")
display(put_expectancy_by_tenor_group_tier_df)

In [ ]:
# ============================================================
# Save diagnostic outputs
# ============================================================

summary_by_side_df.to_csv(AUDIT_DIR / "signal_perf_by_side_v0_1.csv", index=False)
summary_by_side_year_df.to_csv(AUDIT_DIR / "signal_perf_by_side_year_v0_1.csv", index=False)
summary_by_side_tenor_df.to_csv(AUDIT_DIR / "signal_perf_by_side_tenor_v0_1.csv", index=False)
summary_by_side_tenor_group_df.to_csv(AUDIT_DIR / "signal_perf_by_side_tenor_group_v0_1.csv", index=False)

put_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_tier_v0_1.csv", index=False)
put_tenor_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_tenor_tier_v0_1.csv", index=False)
put_tenor_group_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_tenor_group_tier_v0_1.csv", index=False)

put_blended_z_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_blended_z_bucket_v0_1.csv", index=False)
put_primary_vrp_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_primary_vrp_bucket_v0_1.csv", index=False)
put_trailing_rv_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_trailing_rv_bucket_v0_1.csv", index=False)
put_rsi_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_rsi_bucket_v0_1.csv", index=False)

front_vs_nonfront_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_front_vs_nonfront_v0_1.csv", index=False)
front_put_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_tier_v0_1.csv", index=False)
front_put_z_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_z_bucket_v0_1.csv", index=False)
front_put_rv_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_rv_bucket_v0_1.csv", index=False)
front_put_rsi_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_rsi_bucket_v0_1.csv", index=False)

put_loss_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_loss_bucket_v0_1.csv", index=False)
put_tail_loss_df.to_csv(AUDIT_DIR / "signal_perf_worst_50_put_losses_v0_1.csv", index=False)

put_expectancy_by_tenor_df.to_csv(AUDIT_DIR / "signal_perf_put_expectancy_by_tenor_v0_1.csv", index=False)
put_expectancy_by_tier_df.to_csv(AUDIT_DIR / "signal_perf_put_expectancy_by_tier_v0_1.csv", index=False)
put_expectancy_by_tenor_group_tier_df.to_csv(AUDIT_DIR / "signal_perf_put_expectancy_by_tenor_group_tier_v0_1.csv", index=False)

print("Saved signal performance diagnostics to:", AUDIT_DIR)